# RQ1: Which baseline machine learning models best detect financial fraud?

**Research Question:** How do baseline classifiers (Logistic Regression, Random Forest, Decision Tree, KNN, Naive Bayes) compare in detecting fraudulent transactions on the Financial Transactions Dataset?

**Hypothesis:** Tree-based models will outperform linear models due to non-linear fraud patterns.

**Target Variable:** `is_fraud` (binary: 0 = legitimate, 1 = fraud)

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['pdf.fonttype'] = 42
import os, warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB

print('Libraries loaded successfully.')

In [ ]:
# ── Load Dataset ─────────────────────────────────────────────────────────────
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
# ── Read CSV (update path if needed after running cell above) ─────────────────
DATA_PATH = '/kaggle/input/financial-transactions-dataset-for-fraud-detection/financial_fraud_detection_dataset.csv'
df = pd.read_csv(DATA_PATH, nrows=50000)
print('Shape:', df.shape)
df.head(3)

In [ ]:
# ── Preprocessing ────────────────────────────────────────────────────────────
TARGET = 'is_fraud'

# Drop leakage + non-informative columns
DROP_COLS = ['transaction_id', 'timestamp', 'sender_account', 'receiver_account',
             'fraud_type', 'ip_address', 'device_hash']
df.drop(columns=[c for c in DROP_COLS if c in df.columns], inplace=True)

# Encode remaining categoricals
cat_cols = df.select_dtypes(include='object').columns.tolist()
le = LabelEncoder()
for col in cat_cols:
    df[col] = le.fit_transform(df[col].astype(str))

df.dropna(inplace=True)

X = df.drop(columns=[TARGET])
y = df[TARGET]
print('Class distribution:\n', y.value_counts())

# Train / Test split  80 / 20
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)
print('Train size:', X_train.shape, ' | Test size:', X_test.shape)

In [ ]:
# ── Train & Evaluate Models ───────────────────────────────────────────────────
models = {
    'Logistic Regression': (LogisticRegression(max_iter=1000, random_state=42), True),
    'Random Forest':       (RandomForestClassifier(n_estimators=100, random_state=42), False),
    'Decision Tree':       (DecisionTreeClassifier(random_state=42), False),
    'KNN':                 (KNeighborsClassifier(n_neighbors=5), True),
    'Naive Bayes':         (GaussianNB(), True),
}

results = []
for name, (model, use_scaled) in models.items():
    Xtr = X_train_sc if use_scaled else X_train
    Xte = X_test_sc  if use_scaled else X_test
    model.fit(Xtr, y_train)
    preds = model.predict(Xte)
    proba = model.predict_proba(Xte)[:,1] if hasattr(model,'predict_proba') else preds
    results.append({
        'Model':     name,
        'Accuracy':  round(accuracy_score(y_test, preds),4),
        'Precision': round(precision_score(y_test, preds, zero_division=0),4),
        'Recall':    round(recall_score(y_test, preds, zero_division=0),4),
        'F1':        round(f1_score(y_test, preds, zero_division=0),4),
        'AUC-ROC':   round(roc_auc_score(y_test, proba),4),
    })
    print(f'{name:25s} done.')

df_res = pd.DataFrame(results)
df_res

In [ ]:
# ── Save Table ────────────────────────────────────────────────────────────────
df_res.to_csv('table_rq1_baseline_comparison.csv', index=False)
print('Table saved: table_rq1_baseline_comparison.csv')

In [ ]:
# ── Publication-Ready Figure ──────────────────────────────────────────────────
metrics = ['Accuracy','Precision','Recall','F1','AUC-ROC']
models_list = df_res['Model'].tolist()
x = np.arange(len(models_list))
width = 0.15
colors = ['#2196F3','#4CAF50','#FF9800','#E91E63','#9C27B0']

fig, ax = plt.subplots(figsize=(12, 6))
for i, (metric, color) in enumerate(zip(metrics, colors)):
    vals = df_res[metric].values
    bars = ax.bar(x + i*width, vals, width, label=metric, color=color, alpha=0.85, edgecolor='white')

ax.set_xlabel('Model', fontsize=13, fontweight='bold')
ax.set_ylabel('Score', fontsize=13, fontweight='bold')
ax.set_title('RQ1: Baseline Model Performance Comparison\nfor Fraud Detection', fontsize=14, fontweight='bold')
ax.set_xticks(x + width*2)
ax.set_xticklabels(models_list, rotation=15, ha='right', fontsize=10)
ax.set_ylim(0, 1.12)
ax.legend(fontsize=10, loc='upper right')
ax.yaxis.grid(True, linestyle='--', alpha=0.7)
ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig('figure_rq1_baseline_comparison.pdf', bbox_inches='tight', dpi=300)
plt.show()
print('Figure saved: figure_rq1_baseline_comparison.pdf')